In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

def load_oe(path, col_names):
    df = pd.read_csv(path, header=None, names=col_names)
    df = df[pd.to_numeric(df["ts"], errors="coerce").notna()].copy()
    df = df[pd.to_numeric(df[col_names[1]], errors="coerce").notna()].copy()
    df = df.astype({c: float for c in col_names if c != "flag"})
    return df.sort_values("ts").reset_index(drop=True)

def load_oe_all(base_path, p):
    acc       = load_oe(base_path + f"{p}_acc.csv",       ["ts","ax","ay","az","flag"])
    gyro      = load_oe(base_path + f"{p}_gyro.csv",      ["ts","gx","gy","gz","flag"])
    mgnt      = load_oe(base_path + f"{p}_mgnt.csv",      ["ts","mx","my","mz","flag"])
    baro      = load_oe(base_path + f"{p}_baro.csv",      ["ts","baro","flag"])
    bone_acc  = load_oe(base_path + f"{p}_bone_acc.csv",  ["ts","bx","by","bz","flag"])
    ppg       = load_oe(base_path + f"{p}_ppg.csv",       ["ts","ppg_red","ppg_ir","ppg_green","ppg_ambient","flag"])
    skin_temp = load_oe(base_path + f"{p}_skin_temp.csv", ["ts","skin_temp","flag"])
    env_temp  = load_oe(base_path + f"{p}_env_temp.csv",  ["ts","env_temp","flag"])

    merged = acc[["ts","ax","ay","az"]].copy()
    for df, cols in [
        (gyro,      ["ts","gx","gy","gz"]),
        (mgnt,      ["ts","mx","my","mz"]),
        (baro,      ["ts","baro"]),
        (bone_acc,  ["ts","bx","by","bz"]),
        (ppg,       ["ts","ppg_red","ppg_ir","ppg_green","ppg_ambient"]),
        (skin_temp, ["ts","skin_temp"]),
        (env_temp,  ["ts","env_temp"]),
    ]:
        merged = pd.merge_asof(merged, df[cols], on="ts", direction="nearest", tolerance=25000)
    merged["participant"] = p
    return merged

def load_xsens(base_path, participants, xsens_offset):
    xs_all = []
    for p in participants:
        df = pd.read_csv(base_path + f"{p}.csv", low_memory=False)
        df["participant"] = p

        df = df.drop(columns=["Unnamed: 11"], errors="ignore")
        for col in ["Acc_X","Acc_Y","Acc_Z","Gyr_X","Gyr_Y","Gyr_Z","PacketCounter"]:
            df = df[pd.to_numeric(df[col], errors="coerce").notna()]
            df[col] = df[col].astype(float)
        df = df[~((df["Acc_X"]==0) & (df["Acc_Y"]==0) & (df["Acc_Z"]==0))]

        # sort by PacketCounter — always correct order, never overflows
        df = df.sort_values("PacketCounter").reset_index(drop=True)

        # check for real gaps in PacketCounter
        pc_diff = df["PacketCounter"].diff()
        jumps = pc_diff[pc_diff > 1000]  # more than ~33s gap at 30Hz
        if len(jumps) > 0:
            print(f"WARNING: {p} Xsens has jump — trimming")
            df = df[df.index < jumps.index[0]].reset_index(drop=True)

        # compute local time from PacketCounter at 30Hz
        df["t_local"] = (df["PacketCounter"] - df["PacketCounter"].iloc[0]) / 30.0

        duration = df["t_local"].iloc[-1]
        print(f"Xsens {p}: {len(df)} rows, duration {duration:.1f}s")
        xs_all.append(df)

    xs = pd.concat(xs_all, ignore_index=True)
    xs = xs.sort_values(["participant","PacketCounter"]).reset_index(drop=True)

    for p in participants:
        mask = xs["participant"] == p
        xs.loc[mask, "t_video"] = xs.loc[mask, "t_local"] + xsens_offset

    xs = xs.sort_values(["t_video","participant"]).reset_index(drop=True)
    return xs

def label_df(df, time_col, elan, all_tiers, collab_cutoff=None):
    for tier in all_tiers:
        tier_rows = elan[elan["tier"] == tier][["t_start_s","t_end_s","label"]].dropna().reset_index(drop=True)

        is_individual = "_" not in tier and tier != "Whole_Group"
        cutoff = None
        if is_individual and collab_cutoff:
            for p_key, p_cutoff in collab_cutoff.items():
                if p_key.lower() == tier.lower():
                    cutoff = p_cutoff
                    break

        t_sorted = df[[time_col]].copy().reset_index()
        t_sorted = t_sorted.sort_values(time_col)

        result = pd.merge_asof(
            t_sorted,
            tier_rows.sort_values("t_start_s"),
            left_on=time_col,
            right_on="t_start_s",
            direction="backward"
        )
        result["label"] = result.apply(
            lambda r: r["label"] if pd.notna(r["t_end_s"]) and r[time_col] <= r["t_end_s"] else "", axis=1
        )
        result = result.set_index("index")["label"].reindex(df.index)
        result = result.fillna("").astype(str)

        if is_individual and cutoff is not None:
            mask = (result == "") & (df[time_col] < cutoff)
            result[mask] = "individual_building"

        df[tier] = result
        print(f"  {tier}: {(df[tier] != '').sum()} labelled rows")

    return df

Group - 1

In [ ]:
xs = pd.read_csv("/content/drive/MyDrive/thesis/data/group_1/group1_xsens_labelled.csv", low_memory=False)
print(f"Rachel rows: {len(xs[xs['participant']=='rachel'])}")
print(f"Rachel t_video: {xs[xs['participant']=='rachel']['t_video'].min():.1f} to {xs[xs['participant']=='rachel']['t_video'].max():.1f}s")

Rachel rows: 6934
Rachel t_video: -25.0 to 215.5s


In [ ]:
import pandas as pd

elan_raw = pd.read_csv("/content/drive/MyDrive/thesis/data/group_1/elan/Group_1.csv", header=None,
                       names=["tier","_","t_start_hms","t_start_s",
                              "t_end_hms","t_end_s","duration_hms","duration_s","label"])

tier_map = {
    "Arda": "Arda", "Bas": "Bas", "Rachel": "Rachel",
    "Bas_Arda": "Bas_Arda", "Arda_Rachel": "Arda_Rachel",
    "Bas_Rachel": "Bas_Rachel", "Whole_Group": "Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip()

typo_map = {
    "travel_between_units": "traveling_between_units",
    "traveling_betwen_units": "traveling_between_units",
    "trveling_between_units": "traveling_between_units",
    "moving_between_units": "traveling_between_units",
    "inspecting_other_puzzle_pieces": "inspecting_other_pieces",
    "inspecitng_other_puzlle_pieces": "inspecting_other_pieces",
    "inspecting_puzzle_pieces": "inspecting_other_pieces",
    "inspecitng_pieces": "inspecting_pieces",
    "merging_subpieces": "merging_subpiece",
    "merging_sub_pieces": "merging_subpiece",
    "carying_subpiece_with_tray": "carrying_subpiece_with_tray",
    "putting_doen_tray": "putting_down_tray",
    "task_operatinal_convo": "task_operational_convo",
    "task_operation_convo": "task_operational_convo",
    "task_realted_convo": "task_operational_convo",
    "task_related_dialouge": "task_operational_convo",
    "task_related_social_convo": "task_social_convo",
    "picking_up_pieces": "picking_up_piece",
    "buılding_subpiece_together": "building_subpiece_together",
    "searching_for_piece": "searching_piece",
    "delivering_pieceBA": "delivering_piece",
    "delivering_pieceRB": "delivering_piece",
    "delivering_trayBR": "delivering_piece",
    "delivering_target_imageRB": "delivering_piece",
    "picking_up_target_image_fR": "lifting_target_image",
    "returning_target_image_tR": "returning_target_image",
    "synchronaziton_move": "sync_move",
    "synchronizaiton_move": "sync_move",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "inspecting_other_pieces +traveling_between units": ["inspecting_other_pieces","traveling_between_units"],
    "inspecting_target_image + task_operational_convo": ["inspecting_target_image","task_operational_convo"],
    "object_handover + task_operational_convo": ["object_handover","task_operational_convo"],
    "traveling_between_units + inspecting_target_image": ["traveling_between_units","inspecting_target_image"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

collab_cutoff = {"Arda": 1420.0, "Bas": 1160.0, "Rachel": 1360.0}
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv("/content/drive/MyDrive/thesis/data/group_1/elan/Group_1_clean.csv", index=False)
print(f"Group 1 ELAN: {len(elan)} rows, {elan['label'].nunique()} labels")
print("sync_move rows:", (elan['label']=='sync_move').sum())

Group 1 ELAN: 236 rows, 47 labels
sync_move rows: 1


In [ ]:
BASE            = "/content/drive/MyDrive/thesis/data/group_1/"
ELAN_FILE       = BASE + "elan/Group_1_clean.csv"
PARTICIPANTS    = ["arda", "bas", "rachel"]
VIDEO_START_UTC = datetime(2026, 4, 21, 8, 14, 47, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 21, 8, 14, 22, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC

ALL_TIERS = ["Arda", "Bas", "Rachel",
             "Bas_Arda", "Arda_Rachel", "Bas_Rachel", "Whole_Group"]

collab_cutoff = {"arda": 1420.0, "bas": 1160.0, "rachel": 1360.0}

elan = pd.read_csv(ELAN_FILE)

# OE
oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

# Xsens
xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video: {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

# label
print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS, collab_cutoff)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS, collab_cutoff)

# clean NaN → empty string in tier columns
for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group1_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group1_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved — OE: {oe.shape}, Xsens: {xs.shape}")

OE arda: 99929 rows
OE bas: 96731 rows
OE rachel: 98755 rows
Xsens arda: 60390 rows, duration 2013.0s
Xsens bas: 60388 rows, duration 2012.9s
Xsens rachel: 59992 rows, duration 2012.9s

OE t_video: -104.1 to 1897.0s
Xsens t_video: -25.0 to 1988.0s

Labelling OE:
  Arda: 245027 labelled rows
  Bas: 211491 labelled rows
  Rachel: 224903 labelled rows
  Bas_Arda: 3772 labelled rows
  Arda_Rachel: 7114 labelled rows
  Bas_Rachel: 78425 labelled rows
  Whole_Group: 77246 labelled rows

Labelling Xsens:
  Arda: 141462 labelled rows
  Bas: 120981 labelled rows
  Rachel: 129177 labelled rows
  Bas_Arda: 2286 labelled rows
  Arda_Rachel: 4275 labelled rows
  Bas_Rachel: 48051 labelled rows
  Whole_Group: 46693 labelled rows

Saved — OE: (295415, 29), Xsens: (180770, 21)


Group - 2

In [ ]:
elan_raw = pd.read_csv("/content/drive/MyDrive/thesis/data/group_2/elan/Group_2.csv", header=None,
                       names=["tier","_","t_start_hms","t_start_s",
                              "t_end_hms","t_end_s","duration_hms","duration_s","label"])

tier_map = {
    "Khalil": "Participant1", "Arda": "Participant2", "Shizin": "Participant3",
    "Arda_Khalil": "Participant1_Participant2",
    "Arda_Shizin": "Participant2_Participant3",
    "Shizin_Khalil": "Participant1_Participant3",
    "Whole_Group": "Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip().replace(tier_map)

typo_map = {
    "travel_between_units": "traveling_between_units",
    "traveling_betwen_units": "traveling_between_units",
    "travelin_between_units": "traveling_between_units",
    "putting_doen_piece": "putting_down_piece",
    "puting_down_piece": "putting_down_piece",
    "putting_down_piece_fS": "putting_down_piece",
    "putting_down_object": "putting_down_piece",
    "inspecitng_other_pieces": "inspecting_other_pieces",
    "inspecitng_target_image": "inspecting_target_image",
    "inspecting_piece": "inspecting_pieces",
    "co_building_subpiece": "co_building_subpiece",
    "co_building_sub_piece": "co_building_subpiece",
    "co_building_subpart": "co_building_subpiece",
    "co_building_sub_part": "co_building_subpiece",
    "co_building_the_subpiece": "co_building_subpiece",
    "co_builidng_subpiece": "co_building_subpiece",
    "mering_sub_pieces": "merging_subpiece",
    "mering_subpieces": "merging_subpiece",
    "merging_subpieces": "merging_subpiece",
    "carrying_tray_to_centraltable": "carrying_tray_to_table",
    "carrying_tray_to_central_table": "carrying_tray_to_table",
    "placing_sub_piece_to_tray": "placing_subpiece_to_tray",
    "placing_subpieces_to_tray": "placing_subpiece_to_tray",
    "placing_subpiece_to_centraltable": "placing_subpiece_to_tray",
    "placing_subpiece_to_centraltable_from_tray": "placing_subpiece_to_tray",
    "placing_subpiece_from_tray_to_centraltable": "placing_subpiece_to_tray",
    "co_placing_subpiece_to_tray": "placing_subpiece_to_tray",
    "task_operational_talk": "task_operational_convo",
    "matching_pieces_with_image": "matching_pieces_to_target_image",
    "pointing_thowards_target_image": "presenting_target_image",
    "picking_up_pieceS": "picking_up_piece",
    "picking_up_piecefS": "picking_up_piece",
    "picking_up_piece_fS": "picking_up_piece",
    "picking_up_target_imageS": "picking_up_target_image",
    "delivering_pieceS": "delivering_piece",
    "delivering_piece_toS": "delivering_piece",
    "start_of_individual_build": "sync_move",
    "khalil_synchornizaion_move": "sync_move",
    "droping_erarble_syncornaziton_move": "sync_move",
    "synchronaziton_move": "sync_move",
    "synchronizaiton_move": "sync_move",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "picking_up_piece + traveling_between_units": ["picking_up_piece","traveling_between_units"],
    "putting_down_piece + traveling_between_units": ["putting_down_piece","traveling_between_units"],
    "inspecting_target_image + inspecting_pieces": ["inspecting_target_image","inspecting_pieces"],
    "inspecting_pieces + inspecting_target_image": ["inspecting_pieces","inspecting_target_image"],
    "inspecting_target_image + matching_pieces_to_target_image": ["inspecting_target_image","matching_pieces_to_target_image"],
    "traveling_between_units + inspecting_other_pieces": ["traveling_between_units","inspecting_other_pieces"],
    "inspecitng_pieces + task_operational_convo": ["inspecting_pieces","task_operational_convo"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

collab_cutoff = {"Participant1": 1742.0, "Participant2": 1742.0, "Participant3": 1742.0}
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv("/content/drive/MyDrive/thesis/data/group_2/elan/Group_2_clean.csv", index=False)
print(f"Group 2 ELAN: {len(elan)} rows, {elan['label'].nunique()} labels")
print("sync_move rows:", (elan['label']=='sync_move').sum())

Group 2 ELAN: 312 rows, 29 labels
sync_move rows: 3


In [ ]:
BASE            = "/content/drive/MyDrive/thesis/data/group_2/"
ELAN_FILE       = BASE + "elan/Group_2_clean.csv"
PARTICIPANTS    = ["Participant1", "Participant2", "Participant3"]
VIDEO_START_UTC = datetime(2026, 4, 21, 12, 4, 25, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 21, 12, 7, 51, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC

ALL_TIERS = ["Participant1", "Participant2", "Participant3",
             "Participant1_Participant2", "Participant1_Participant3",
             "Participant2_Participant3", "Whole_Group"]

collab_cutoff = {"Participant1": 1742.0, "Participant2": 1742.0, "Participant3": 1742.0}

elan = pd.read_csv(ELAN_FILE)

oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video: {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS)

# clean NaN → empty string in tier columns
for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group2_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group2_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved — OE: {oe.shape}, Xsens: {xs.shape}")

OE Participant1: 142007 rows
OE Participant2: 132547 rows
OE Participant3: 123936 rows
Xsens Participant1: 83543 rows, duration 2784.7s
Xsens Participant2: 83543 rows, duration 2784.7s
Xsens Participant3: 83544 rows, duration 2784.8s

OE t_video: 51.0 to 2907.4s
Xsens t_video: 206.0 to 2990.8s

Labelling OE:
  Participant1: 250613 labelled rows
  Participant2: 243837 labelled rows
  Participant3: 240549 labelled rows
  Participant1_Participant2: 0 labelled rows
  Participant1_Participant3: 0 labelled rows
  Participant2_Participant3: 0 labelled rows
  Whole_Group: 0 labelled rows

Labelling Xsens:
  Participant1: 146844 labelled rows
  Participant2: 142869 labelled rows
  Participant3: 140832 labelled rows
  Participant1_Participant2: 0 labelled rows
  Participant1_Participant3: 0 labelled rows
  Participant2_Participant3: 0 labelled rows
  Whole_Group: 0 labelled rows

Saved — OE: (398490, 29), Xsens: (250630, 21)


Group - 3

In [ ]:
p1 = pd.read_csv("/content/drive/MyDrive/thesis/data/group_3/elan/Group_3_Part1.csv", header=None,
                 names=["tier","_","t_start_hms","t_start_s","t_end_hms","t_end_s","duration_hms","duration_s","label"])
p2 = pd.read_csv("/content/drive/MyDrive/thesis/data/group_3/elan/Group_3_Part2.csv", header=None,
                 names=["tier","_","t_start_hms","t_start_s","t_end_hms","t_end_s","duration_hms","duration_s","label"])

PART2_OFFSET = 1506.0
p2["t_start_s"] = p2["t_start_s"] + PART2_OFFSET
p2["t_end_s"]   = p2["t_end_s"]   + PART2_OFFSET
elan_raw = pd.concat([p1, p2], ignore_index=True)

tier_map = {
    "chinyu": "Participant1", "long": "Participant2", "egemen": "Participant3",
    "egemen chinyu": "Participant1_Participant3",
    "egemen long": "Participant2_Participant3",
    "whole group": "Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip().replace(tier_map)

typo_map = {
    "traveling_betwwen_units": "traveling_between_units",
    "traveling_between_unit": "traveling_between_units",
    "inspecting_piece": "inspecting_pieces",
    "inspecitng_pieces": "inspecting_pieces",
    "picking_up_targetimage": "picking_up_target_image",
    "approaching_to_central_table": "traveling_between_units",
    "approaching_to_single_units": "traveling_between_units",
    "inspecting_setup": "inspecting_target_image",
    "co_building_subpiece": "co_building_subpiece",
    "co_building_sub_piece": "co_building_subpiece",
    "merging_sub_piece": "merging_subpiece",
    "mering_subpieces": "merging_subpiece",
    "moving_subpiece_from_tray_to_central_table": "moving_pieces_to_central_table_from_tray",
    "moving_subpiece_to_central_table_from_tray": "moving_pieces_to_central_table_from_tray",
    "moving_subpiece_to_tray": "placing_subpiece_to_tray",
    "moving_tray_to_central_table": "carrying_tray_to_table",
    "mathicng_pieces_to_target_image": "matching_pieces_to_target_image",
    "task_operational_talk": "task_operational_convo",
    "task_operatşonal_convo": "task_operational_convo",
    "pointing_to_piece": "presenting_piece",
    "synchronaziton_move": "sync_move",
    "synchronizaiton_move": "sync_move",
    "synchronaztion_move": "sync_move",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "approaching_to_central_table + task_operational_convo": ["traveling_between_units","task_operational_convo"],
    "task_operational_convo + isnpecting_setup": ["task_operational_convo","inspecting_target_image"],
    "picking_up_target_image + traveling_between_units": ["picking_up_target_image","traveling_between_units"],
    "object_handover + task_operational_talk": ["object_handover","task_operational_convo"],
    "putting_down_target_image + traveling_between_units": ["putting_down_target_image","traveling_between_units"],
    "putting_down_target_image + traveling_betwwen_units": ["putting_down_target_image","traveling_between_units"],
    "traveling_between_units + putting_down_piece": ["traveling_between_units","putting_down_piece"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

collab_cutoff = {"Participant1": 2495.0, "Participant2": 2495.0, "Participant3": 2065.0}
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv("/content/drive/MyDrive/thesis/data/group_3/elan/Group_3_clean.csv", index=False)
print(f"Group 3 ELAN: {len(elan)} rows, {elan['label'].nunique()} labels")
print("sync_move rows:", (elan['label']=='sync_move').sum())

Group 3 ELAN: 170 rows, 23 labels
sync_move rows: 1


In [ ]:
import pandas as pd

for part, path in [
    ("Part 1", "/content/drive/MyDrive/thesis/data/group_3/elan/Group_3_Part1.csv"),
    ("Part 2", "/content/drive/MyDrive/thesis/data/group_3/elan/Group_3_Part2.csv"),
]:
    df = pd.read_csv(path, header=None,
                     names=["tier","_","t_start_hms","t_start_s",
                            "t_end_hms","t_end_s","duration_hms","duration_s","label"])
    print(f"=== {part} ===")
    print(f"Rows: {len(df)}")
    print("All labels:")
    for l in sorted(df["label"].dropna().str.strip().unique().tolist()):
        print(f"  {l}")
    print()

=== Part 1 ===
Rows: 40
All labels:
  approaching_to_central_table + task_operational_convo
  approaching_to_single_units
  inspecting_other_pieces
  inspecting_piece
  inspecting_setup
  inspecting_target_image
  object_handover
  picking_up_target_image
  picking_up_target_image + traveling_between_units
  picking_up_targetimage
  presenting_piece
  putting_down_target_image
  searching_target_image
  synchronaztion_move
  task_operational_convo
  task_operational_convo + isnpecting_setup
  task_social_convo
  traveling_between_units

=== Part 2 ===
Rows: 80
All labels:
  co_building_sub_piece
  co_building_subpiece
  delivering_piece
  inspecting_other_pieces
  inspecting_pieces
  mathicng_pieces_to_target_image
  merging_sub_piece
  mering_subpieces
  moving_pieces_to_central_table_from_tray
  moving_subpiece_from_tray_to_central_table
  moving_subpiece_to_central_table_from_tray
  moving_subpiece_to_tray
  moving_tray_to_central_table
  object_handover
  object_handover + task_ope

In [ ]:
BASE            = "/content/drive/MyDrive/thesis/data/group_3/"
ELAN_FILE       = BASE + "elan/Group_3_clean.csv"
PARTICIPANTS    = ["Participant1", "Participant2", "Participant3"]
VIDEO_START_UTC = datetime(2026, 4, 22, 12, 21, 50, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 22, 12, 23, 31, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC

ALL_TIERS = ["Participant1", "Participant2", "Participant3",
             "Participant1_Participant2", "Participant1_Participant3",
             "Participant2_Participant3", "Whole_Group"]

collab_cutoff = {"Participant1": 2495.0, "Participant2": 2495.0, "Participant3": 2065.0}

elan = pd.read_csv(ELAN_FILE)

oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video: {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS)

# clean NaN → empty string in tier columns
for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group3_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group3_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved — OE: {oe.shape}, Xsens: {xs.shape}")

OE Participant1: 123862 rows
OE Participant2: 127912 rows
OE Participant3: 137760 rows
Xsens Participant1: 82976 rows, duration 2765.9s
Xsens Participant2: 82977 rows, duration 2765.9s
Xsens Participant3: 82978 rows, duration 2765.9s

OE t_video: 23.5 to 2782.6s
Xsens t_video: 101.0 to 2866.9s

Labelling OE:
  Participant1: 349607 labelled rows
  Participant2: 349025 labelled rows
  Participant3: 290811 labelled rows
  Participant1_Participant2: 0 labelled rows
  Participant1_Participant3: 65495 labelled rows
  Participant2_Participant3: 20032 labelled rows
  Whole_Group: 66886 labelled rows

Labelling Xsens:
  Participant1: 216368 labelled rows
  Participant2: 216005 labelled rows
  Participant3: 179540 labelled rows
  Participant1_Participant2: 0 labelled rows
  Participant1_Participant3: 41016 labelled rows
  Participant2_Participant3: 12771 labelled rows
  Whole_Group: 39090 labelled rows

Saved — OE: (389534, 29), Xsens: (248931, 21)


Group - 5

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/thesis/data/group_5/xsens/Participant1.csv", low_memory=False)
print(df["PacketCounter"].head(10).tolist())
print(df["PacketCounter"].tail(10).tolist())
print(f"PacketCounter min: {df['PacketCounter'].min()}, max: {df['PacketCounter'].max()}")
print(f"SampleTimeFine first 5: {df['SampleTimeFine'].head(5).tolist()}")
print(f"SampleTimeFine last 5: {df['SampleTimeFine'].tail(5).tolist()}")

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[121035, 121036, 121037, 121038, 121039, 121040, 121041, 121042, 121043, 121044]
PacketCounter min: 0, max: 121044
SampleTimeFine first 5: [2902382686, 2902416020, 2902449354, 2902482688, 2902516022]
SampleTimeFine last 5: [2642162750, 2642196084, 2642229418, 2642262752, 2642296086]


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/thesis/data/group_5/xsens/Participant1.csv", low_memory=False)
df["SampleTimeFine"] = pd.to_numeric(df["SampleTimeFine"], errors="coerce")
df["PacketCounter"] = pd.to_numeric(df["PacketCounter"], errors="coerce")
df = df.dropna(subset=["SampleTimeFine","PacketCounter"])
df["SampleTimeFine"] = df["SampleTimeFine"].astype(float)
df = df.sort_values("PacketCounter").reset_index(drop=True)

# apply overflow fix
MAX_32 = 4294967296
stf = df["SampleTimeFine"].values.copy()
offset = 0
for i in range(1, len(stf)):
    if stf[i] + offset < stf[i-1] + offset - 1e8:
        offset += MAX_32
    stf[i] += offset
df["SampleTimeFine"] = stf

# find jumps
diff = df["SampleTimeFine"].diff()
jumps = diff[diff > 1e8]
print(f"Jumps after fix: {len(jumps)}")
for idx, val in jumps.items():
    print(f"  idx={idx}, gap={val/1e6:.1f}s")
    print(f"  PacketCounter around jump: {df.loc[idx-1,'PacketCounter']} → {df.loc[idx,'PacketCounter']}")
    print(f"  SampleTimeFine: {df.loc[idx-1,'SampleTimeFine']:.0f} → {df.loc[idx,'SampleTimeFine']:.0f}")

Streaming output truncated to the last 5000 lines.
  PacketCounter around jump: 119377 → 119378
  SampleTimeFine: 494357617465204 → 494361912465834
  idx=119379, gap=4295.0s
  PacketCounter around jump: 119378 → 119379
  SampleTimeFine: 494361912465834 → 494366207466464
  idx=119380, gap=4295.0s
  PacketCounter around jump: 119379 → 119380
  SampleTimeFine: 494366207466464 → 494370502467094
  idx=119381, gap=4295.0s
  PacketCounter around jump: 119380 → 119381
  SampleTimeFine: 494370502467094 → 494374797467724
  idx=119382, gap=4295.0s
  PacketCounter around jump: 119381 → 119382
  SampleTimeFine: 494374797467724 → 494379092468354
  idx=119383, gap=4295.0s
  PacketCounter around jump: 119382 → 119383
  SampleTimeFine: 494379092468354 → 494383387468984
  idx=119384, gap=4295.0s
  PacketCounter around jump: 119383 → 119384
  SampleTimeFine: 494383387468984 → 494387682469614
  idx=119385, gap=4295.0s
  PacketCounter around jump: 119384 → 119385
  SampleTimeFine: 494387682469614 → 4943919

In [ ]:
elan_raw = pd.read_csv("/content/drive/MyDrive/thesis/data/group_5/elan/Group_5.csv", header=None,
                       names=["tier","_","t_start_hms","t_start_s",
                              "t_end_hms","t_end_s","duration_hms","duration_s","label"])

tier_map = {
    "Mintan": "Participant1", "Adarsh": "Participant2", "Ali": "Participant3",
    "Adarsh Mintan": "Participant1_Participant2",
    "Mintan Ali": "Participant1_Participant3",
    "Ali Adarsh": "Participant2_Participant3",
    "Whole Group": "Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip().replace(tier_map)

typo_map = {
    "traveling_between__units": "traveling_between_units",
    "traveling_between_untis": "traveling_between_units",
    "traveling_betwen_units": "traveling_between_units",
    "trvaeling_between_units": "traveling_between_units",
    "approcahing_to_piece": "traveling_between_units",
    "moving_to_single_units": "traveling_between_units",
    "pickinf_up_pieces": "picking_up_piece",
    "picking_up_pieces": "picking_up_piece",
    "pıcking_up_image": "picking_up_target_image",
    "picking_up_image": "picking_up_target_image",
    "putting_doen_tray": "putting_down_tray",
    "putting_down_image": "putting_down_target_image",
    "putting_down_tagret_image": "putting_down_target_image",
    "inscpecing_other_pieces": "inspecting_other_pieces",
    "looking_target_image": "inspecting_target_image",
    "matchin_pieces_with_target_image": "matching_pieces_to_target_image",
    "matching_image_to_pieces": "matching_pieces_to_target_image",
    "matching_pieces_to_image": "matching_pieces_to_target_image",
    "matching_pieces_with_image": "matching_pieces_to_target_image",
    "matching_pieces_with_target_image": "matching_pieces_to_target_image",
    "mathing_pieces_with_image": "matching_pieces_to_target_image",
    "merging_sub_pieces": "merging_subpiece",
    "moving_pieces_from_tray_to_central_table": "moving_pieces_to_central_table_from_tray",
    "moving_pieces_to_table__from_tray": "moving_pieces_to_central_table_from_tray",
    "moving_subpieces_to_tray": "moving_pieces_to_tray",
    "search_for_target_image": "searching_target_image",
    "searchin_for_image": "searching_target_image",
    "searching_for_image": "searching_target_image",
    "searching_for_target_image": "searching_target_image",
    "searching_tray": "searching_piece",
    "task_operaitonal_convo": "task_operational_convo",
    "pointing_to_piece": "presenting_piece",
    "replacing_image": "returning_target_image",
    "co_buiilding_subpiece": "co_building_subpiece",
    "co_building_pieces": "co_building_subpiece",
    "co_building_sub_piece": "co_building_subpiece",
    "co_building_sub_pieec": "co_building_subpiece",
    "co_builidng_sub_piece": "co_building_subpiece",
    "co_builidng_subpiece": "co_building_subpiece",
    "co_merging_sub_pieces": "co_merging_subpiece",
    "co_merging_subpieces": "co_merging_subpiece",
    "co_inspceitng_image": "co_inspecting_image",
    "co_inspecitng_image": "co_inspecting_image",
    "clap_synchronizaiton_move": "sync_move",
    "synchronaziton_move": "sync_move",
    "synchronizaiton_move": "sync_move",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "co_building_subpiece + task_operational_convo": ["co_building_subpiece","task_operational_convo"],
    "co_inspecting_target_image + task_operational_convo": ["co_inspecting_image","task_operational_convo"],
    "object_handover + co_inspecting_image": ["object_handover","co_inspecting_image"],
    "object_handover + task_operational convo": ["object_handover","task_operational_convo"],
    "object_handover + task_operational_convo": ["object_handover","task_operational_convo"],
    "picking_up_image +traveling_between_units": ["picking_up_target_image","traveling_between_units"],
    "picking_up_target_image + pickinig_up_tray": ["picking_up_target_image","picking_up_tray"],
    "picking_up_target_image + traveling_between_units": ["picking_up_target_image","traveling_between_units"],
    "task_operational_convo + inspecting_subpiece": ["task_operational_convo","inspecting_other_pieces"],
    "traveling_between_units + presenting_target_image": ["traveling_between_units","presenting_target_image"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

collab_cutoff = {"Participant1": 2965.0, "Participant2": 2047.0, "Participant3": 2047.0}
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv("/content/drive/MyDrive/thesis/data/group_5/elan/Group_5_clean.csv", index=False)
print(f"Group 5 ELAN: {len(elan)} rows, {elan['label'].nunique()} labels")
print("sync_move rows:", (elan['label']=='sync_move').sum())

Group 5 ELAN: 373 rows, 31 labels
sync_move rows: 3


In [ ]:
BASE            = "/content/drive/MyDrive/thesis/data/group_5/"
ELAN_FILE       = BASE + "elan/Group_5_clean.csv"
PARTICIPANTS    = ["Participant1", "Participant2", "Participant3"]
VIDEO_START_UTC = datetime(2026, 4, 23, 11, 8, 44, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 23, 11, 12, 10, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC

ALL_TIERS = ["Participant1", "Participant2", "Participant3",
             "Participant1_Participant2", "Participant1_Participant3",
             "Participant2_Participant3", "Whole_Group"]

collab_cutoff = {"Participant1": 2965.0, "Participant2": 2047.0, "Participant3": 2047.0}

elan = pd.read_csv(ELAN_FILE)

oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video: {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS)

# clean NaN → empty string in tier columns
for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group5_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group5_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved — OE: {oe.shape}, Xsens: {xs.shape}")

OE Participant1: 171863 rows
OE Participant2: 190456 rows
OE Participant3: 200671 rows
Xsens Participant1: 121043 rows, duration 4034.8s
Xsens Participant2: 121043 rows, duration 4034.7s
Xsens Participant3: 121044 rows, duration 4034.8s

OE t_video: 123.3 to 4152.1s
Xsens t_video: 206.0 to 4240.8s

Labelling OE:
  Participant1: 413980 labelled rows
  Participant2: 285061 labelled rows
  Participant3: 281931 labelled rows
  Participant1_Participant2: 15100 labelled rows
  Participant1_Participant3: 7550 labelled rows
  Participant2_Participant3: 125236 labelled rows
  Whole_Group: 142359 labelled rows

Labelling Xsens:
  Participant1: 261945 labelled rows
  Participant2: 178986 labelled rows
  Participant3: 177183 labelled rows
  Participant1_Participant2: 9537 labelled rows
  Participant1_Participant3: 5007 labelled rows
  Participant2_Participant3: 80706 labelled rows
  Whole_Group: 89069 labelled rows

Saved — OE: (562990, 29), Xsens: (363130, 21)


In [ ]:
import pandas as pd

groups = {
    "group_1": {
        "oe":    "/content/drive/MyDrive/thesis/data/group_1/group1_oe_labelled.csv",
        "xsens": "/content/drive/MyDrive/thesis/data/group_1/group1_xsens_labelled.csv",
        "tiers": ["Arda","Bas","Rachel","Bas_Arda","Arda_Rachel","Bas_Rachel","Whole_Group"],
    },
    "group_2": {
        "oe":    "/content/drive/MyDrive/thesis/data/group_2/group2_oe_labelled.csv",
        "xsens": "/content/drive/MyDrive/thesis/data/group_2/group2_xsens_labelled.csv",
        "tiers": ["Participant1","Participant2","Participant3",
                  "Participant1_Participant2","Participant1_Participant3",
                  "Participant2_Participant3","Whole_Group"],
    },
    "group_3": {
        "oe":    "/content/drive/MyDrive/thesis/data/group_3/group3_oe_labelled.csv",
        "xsens": "/content/drive/MyDrive/thesis/data/group_3/group3_xsens_labelled.csv",
        "tiers": ["Participant1","Participant2","Participant3",
                  "Participant1_Participant2","Participant1_Participant3",
                  "Participant2_Participant3","Whole_Group"],
    },
    "group_5": {
        "oe":    "/content/drive/MyDrive/thesis/data/group_5/group5_oe_labelled.csv",
        "xsens": "/content/drive/MyDrive/thesis/data/group_5/group5_xsens_labelled.csv",
        "tiers": ["Participant1","Participant2","Participant3",
                  "Participant1_Participant2","Participant1_Participant3",
                  "Participant2_Participant3","Whole_Group"],
    },
}

for group, info in groups.items():
    print(f"\n{'='*40}")
    print(f"{group.upper()}")
    for sensor, path in [("OE", info["oe"]), ("Xsens", info["xsens"])]:
        df = pd.read_csv(path, low_memory=False, keep_default_na=False)
        tiers = info["tiers"]

        # 1. sync_move present
        sync_counts = {t: (df[t] == "sync_move").sum() for t in tiers}
        sync_total  = sum(sync_counts.values())

        # 2. any NaN in tier columns
        nan_counts  = {t: df[t].isna().sum() for t in tiers}
        nan_total   = sum(nan_counts.values())

        # 3. individual_building overlapping sync_move rows
        overlap = 0
        for t in tiers:
            if "_" not in t and t != "Whole_Group":  # individual tiers only
                # find rows where another tier has sync_move and this tier has individual_building
                sync_mask = df[[x for x in tiers if x != t]].apply(
                    lambda col: col == "sync_move", axis=0).any(axis=1)
                ib_mask = df[t] == "individual_building"
                overlap += (sync_mask & ib_mask).sum()

        print(f"  {sensor}: shape={df.shape} | "
              f"sync_move_rows={sync_total} | "
              f"NaNs={nan_total} | "
              f"individual_building+sync_overlap={overlap}")
        if sync_total == 0:
            print(f"    ⚠️  NO sync_move found — needs reprocessing")
        if nan_total > 0:
            print(f"    ⚠️  NaNs: {nan_counts}")
        if overlap > 0:
            print(f"    ⚠️  overlap in tiers: check individual_building gap filling")


GROUP_1
  OE: shape=(295415, 29) | sync_move_rows=346 | NaNs=0 | individual_building+sync_overlap=0
  Xsens: shape=(180770, 21) | sync_move_rows=222 | NaNs=0 | individual_building+sync_overlap=0

GROUP_2
  OE: shape=(398490, 29) | sync_move_rows=1767 | NaNs=0 | individual_building+sync_overlap=0
  Xsens: shape=(250630, 21) | sync_move_rows=783 | NaNs=0 | individual_building+sync_overlap=6
    ⚠️  overlap in tiers: check individual_building gap filling

GROUP_3
  OE: shape=(389534, 29) | sync_move_rows=596 | NaNs=0 | individual_building+sync_overlap=0
  Xsens: shape=(248931, 21) | sync_move_rows=369 | NaNs=0 | individual_building+sync_overlap=9
    ⚠️  overlap in tiers: check individual_building gap filling

GROUP_5
  OE: shape=(562990, 29) | sync_move_rows=1359 | NaNs=0 | individual_building+sync_overlap=0
  Xsens: shape=(363130, 21) | sync_move_rows=843 | NaNs=0 | individual_building+sync_overlap=0


In [ ]:
import pandas as pd

for group, path, tiers in [
    ("group_2", "/content/drive/MyDrive/thesis/data/group_2/group2_xsens_labelled.csv",
     ["Participant1","Participant2","Participant3"]),
    ("group_3", "/content/drive/MyDrive/thesis/data/group_3/group3_xsens_labelled.csv",
     ["Participant1","Participant2","Participant3"]),
]:
    df = pd.read_csv(path, low_memory=False, keep_default_na=False)
    all_tiers = [c for c in df.columns if "Participant" in c or c == "Whole_Group"]
    for t in tiers:
        sync_mask = df[[x for x in all_tiers if x != t]].apply(
            lambda col: col == "sync_move", axis=0).any(axis=1)
        ib_mask = df[t] == "individual_building"
        overlap = df[sync_mask & ib_mask]
        if len(overlap) > 0:
            print(f"{group} — {t}: {len(overlap)} overlap rows")
            print(f"  t_video range: {overlap['t_video'].min():.2f} to {overlap['t_video'].max():.2f}s")
            print(f"  sync_move in tiers: {[x for x in all_tiers if x != t and (overlap[x]=='sync_move').any()]}")

group_2 — Participant2: 3 overlap rows
  t_video range: 295.40 to 295.40s
  sync_move in tiers: ['Participant1']
group_2 — Participant3: 3 overlap rows
  t_video range: 295.40 to 295.40s
  sync_move in tiers: ['Participant1']
group_3 — Participant1: 3 overlap rows
  t_video range: 275.00 to 275.00s
  sync_move in tiers: ['Whole_Group']
group_3 — Participant2: 3 overlap rows
  t_video range: 275.00 to 275.00s
  sync_move in tiers: ['Whole_Group']
group_3 — Participant3: 3 overlap rows
  t_video range: 275.00 to 275.00s
  sync_move in tiers: ['Whole_Group']


GROUP_6

In [ ]:
from datetime import datetime, timezone
import pandas as pd

BASE = "/content/drive/MyDrive/thesis/data/group_6/openearable/"
for p in ["Participant1", "Participant2", "Participant3"]:
    df = pd.read_csv(f"{BASE}{p}/{p}_acc.csv", header=None)
    ts = float(df.iloc[0, 0]) / 1e6
    dt = datetime.fromtimestamp(ts, tz=timezone.utc)
    print(f"{p}: {dt.strftime('%H:%M:%S')} UTC")

Participant1: 14:07:45 UTC
Participant2: 14:07:40 UTC
Participant3: 14:07:32 UTC


In [ ]:
elan = pd.read_csv("/content/drive/MyDrive/thesis/data/group_6/elan/Group_6.csv", header=None)
print("Tiers:", elan[0].unique().tolist())
print("\nLabels:")
for l in sorted(elan[8].dropna().str.strip().unique().tolist()):
    print(f"  {l}")

Tiers: ['Mark', 'Gozluk', 'Kas', 'Mark Gozluk', 'Gozluk Kas', 'Kas Mark', 'Whole Group']

Labels:
  building_subpiece
  carrying_tray
  caryying_tray
  co_building_sub_piece
  co_building_subpiece
  co_carrying_tray
  inspecitng_other_unit
  inspecitng_other_units
  inspecting_other_units
  inspecting_others_unit
  inspecting_piece
  inspecting_pieces
  inspecting_target_image
  matching_pieces_to_target_image
  merging_sub_pieces
  moving_pieces_from_table_to_tray
  moving_pieces_from_tray_to_central_table
  moving_pieces_to_tray_from_table
  non_task_convo
  object_handover
  object_handover + task_operational_convo
  picking_up_piece
  picking_up_pieces
  picking_up_target_image
  picking_up_target_image + traveling_between_units
  picking_up_tray
  pickinup_piece
  presenting_piece
  presenting_target_image
  putting_down_piece
  putting_down_target_image
  putting_down_target_image + traveling_between_units
  putting_down_tray
  searching_for_target_image
  searching_piece
  searc

In [ ]:
elan = pd.read_csv("/content/drive/MyDrive/thesis/data/group_1/elan/Group_1_clean.csv")

# check what's annotated around the sync move time
sync = elan[elan["label"] == "sync_move"]
print("Sync move rows:")
print(sync[["tier","t_start_s","t_end_s","label"]].to_string())

print("\nCollab cutoffs were: arda=1420, bas=1160, rachel=1360")
print("Sync move after all cutoffs?", sync["t_start_s"].min() > 1420)

Sync move rows:
Empty DataFrame
Columns: [tier, t_start_s, t_end_s, label]
Index: []

Collab cutoffs were: arda=1420, bas=1160, rachel=1360
Sync move after all cutoffs? False


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

BASE         = "/content/drive/MyDrive/thesis/data/group_6/"
ELAN_FILE    = BASE + "elan/Group_6_clean.csv"
PARTICIPANTS = ["Participant1", "Participant2", "Participant3"]

VIDEO_START_UTC = datetime(2026, 4, 23, 14, 17, 43, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 23, 14,  9, 38, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC

ALL_TIERS = ["Participant1", "Participant2", "Participant3",
             "Participant1_Participant2", "Participant1_Participant3",
             "Participant2_Participant3", "Whole_Group"]

collab_cutoff = {
    "Participant1": 752.0,
    "Participant2": 1030.0,
    "Participant3": 752.0,
}

# ── ELAN cleaning ─────────────────────────────────────────────────────────────
elan_raw = pd.read_csv(BASE + "elan/Group_6.csv", header=None,
                       names=["tier","_","t_start_hms","t_start_s",
                              "t_end_hms","t_end_s","duration_hms","duration_s","label"])

tier_map = {
    "Kas":         "Participant1",
    "Gozluk":      "Participant2",
    "Mark":        "Participant3",
    "Gozluk Kas":  "Participant1_Participant2",
    "Kas Mark":    "Participant1_Participant3",
    "Mark Gozluk": "Participant2_Participant3",
    "Whole Group": "Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip().replace(tier_map)

typo_map = {
    "traveling_betwen_units":                       "traveling_between_units",
    "inspecitng_other_unit":                        "inspecting_other_units",
    "inspecitng_other_units":                       "inspecting_other_units",
    "inspecting_others_unit":                       "inspecting_other_units",
    "inspecting_piece":                             "inspecting_pieces",
    "pickinup_piece":                               "picking_up_piece",
    "picking_up_pieces":                            "picking_up_piece",
    "caryying_tray":                                "carrying_tray",
    "co_building_sub_piece":                        "co_building_subpiece",
    "merging_sub_pieces":                           "merging_subpiece",
    "moving_pieces_from_table_to_tray":             "moving_pieces_to_tray",
    "moving_pieces_to_tray_from_table":             "moving_pieces_to_tray",
    "moving_pieces_from_tray_to_central_table":     "moving_pieces_to_central_table_from_tray",
    "building_subpiece":                            "individual_building",
    "searching_for_target_image":                   "searching_target_image",
    "task_operaitonal_convo":                       "task_operational_convo",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "object_handover + task_operational_convo":                 ["object_handover","task_operational_convo"],
    "picking_up_target_image + traveling_between_units":        ["picking_up_target_image","traveling_between_units"],
    "putting_down_target_image + traveling_between_units":      ["putting_down_target_image","traveling_between_units"],
    "task_operational_convo + matching_pieces_to_target_image": ["task_operational_convo","matching_pieces_to_target_image"],
    "traveling_between_units + inspecting_other_units":         ["traveling_between_units","inspecting_other_units"],
    "traveling_between_units + putting_down_piece":             ["traveling_between_units","putting_down_piece"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

sync_map = {
    "synchronaziton_move":               "sync_move",
    "synchronizaiton_move":              "sync_move",
    "clap_synchronizaiton_move":         "sync_move",
    "khalil_synchornizaion_move":        "sync_move",
    "droping_erarble_syncornaziton_move":"sync_move",
}
elan_raw["label"] = elan_raw["label"].replace(sync_map)

# individual_building gaps with sync_move exclusion
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv(ELAN_FILE, index=False)
print(f"ELAN saved: {len(elan)} rows, {elan['label'].nunique()} unique labels")
print(sorted(elan["label"].dropna().unique().tolist()))

# ── OE ────────────────────────────────────────────────────────────────────────
oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

# ── Xsens ─────────────────────────────────────────────────────────────────────
xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video:    {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

# ── label ─────────────────────────────────────────────────────────────────────
print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS, collab_cutoff)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS, collab_cutoff)

for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group6_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group6_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved group6_oe_labelled.csv    {oe.shape}")
print(f"Saved group6_xsens_labelled.csv {xs.shape}")

ELAN saved: 164 rows, 27 unique labels
['carrying_tray', 'co_building_subpiece', 'co_carrying_tray', 'individual_building', 'inspecting_other_units', 'inspecting_pieces', 'inspecting_target_image', 'matching_pieces_to_target_image', 'merging_subpiece', 'moving_pieces_to_central_table_from_tray', 'moving_pieces_to_tray', 'non_task_convo', 'object_handover', 'picking_up_piece', 'picking_up_target_image', 'picking_up_tray', 'presenting_piece', 'presenting_target_image', 'putting_down_piece', 'putting_down_target_image', 'putting_down_tray', 'searching_piece', 'searching_target_image', 'sync_move', 'task_operational_convo', 'task_social_convo', 'traveling_between_units']
OE Participant1: 100717 rows
OE Participant2: 97305 rows
OE Participant3: 101315 rows
Xsens Participant1: 59544 rows, duration 1984.8s
Xsens Participant2: 59544 rows, duration 1984.8s
Xsens Participant3: 59542 rows, duration 1984.7s

OE t_video:    -610.3 to 1420.2s
Xsens t_video: -485.0 to 1499.8s

Labelling OE:
  Partici

In [ ]:
import pandas as pd

BASE  = "/content/drive/MyDrive/thesis/data/group_6/"
TIERS = ["Participant1","Participant2","Participant3",
         "Participant1_Participant2","Participant1_Participant3",
         "Participant2_Participant3","Whole_Group"]

for sensor, path in [
    ("OE",    BASE + "group6_oe_labelled.csv"),
    ("Xsens", BASE + "group6_xsens_labelled.csv"),
]:
    df = pd.read_csv(path, low_memory=False, keep_default_na=False)
    print(f"=== GROUP 6 {sensor} ===")
    print(f"Shape:        {df.shape}")
    print(f"Participants: {df['participant'].value_counts().to_dict()}")
    print(f"t_video:      {df['t_video'].min():.1f} to {df['t_video'].max():.1f}s")

    # NaNs
    nan_total = sum(df[t].isna().sum() for t in TIERS)
    print(f"NaNs:         {nan_total}")

    # sync_move
    sync_counts = {t: (df[t]=='sync_move').sum() for t in TIERS}
    print(f"sync_move:    {sync_counts}")

    # individual_building + sync_move overlap
    overlap = 0
    for t in ["Participant1","Participant2","Participant3"]:
        sync_mask = df[[x for x in TIERS if x != t]].apply(
            lambda col: col == "sync_move", axis=0).any(axis=1)
        ib_mask = df[t] == "individual_building"
        overlap += (sync_mask & ib_mask).sum()
    print(f"IB+sync overlap: {overlap}")

    # filled rows per tier
    print("Filled per tier:")
    for t in TIERS:
        filled = (df[t] != '').sum()
        top2   = df[df[t]!=''][t].value_counts().head(2).to_dict()
        print(f"  {t}: {filled} rows — top: {top2}")

    print()

=== GROUP 6 OE ===
Shape:        (299337, 29)
Participants: {'Participant3': 101315, 'Participant1': 100717, 'Participant2': 97305}
t_video:      -610.3 to 1420.2s
NaNs:         0
sync_move:    {'Participant1': np.int64(0), 'Participant2': np.int64(0), 'Participant3': np.int64(0), 'Participant1_Participant2': np.int64(0), 'Participant1_Participant3': np.int64(0), 'Participant2_Participant3': np.int64(0), 'Whole_Group': np.int64(831)}
IB+sync overlap: 864
Filled per tier:
  Participant1: 220565 rows — top: {'individual_building': 205985, 'moving_pieces_to_tray': 3774}
  Participant2: 268808 rows — top: {'individual_building': 253170, 'traveling_between_units': 3779}
  Participant3: 248862 rows — top: {'individual_building': 238617, 'moving_pieces_to_tray': 3497}
  Participant1_Participant2: 2294 rows — top: {'object_handover': 1733, 'task_operational_convo': 561}
  Participant1_Participant3: 54670 rows — top: {'co_building_subpiece': 37742, 'task_operational_convo': 7840}
  Participant2

GROUP_7


In [15]:
from datetime import datetime, timezone
import pandas as pd

BASE = "/content/drive/MyDrive/thesis/data/group_7/openearable/"
for p in ["Participant1", "Participant2", "Participant3"]:
    df = pd.read_csv(f"{BASE}{p}/{p}_acc.csv", header=None)
    ts = float(df.iloc[0, 0]) / 1e6
    dt = datetime.fromtimestamp(ts, tz=timezone.utc)
    print(f"{p}: {dt.strftime('%H:%M:%S')} UTC")

Participant1: 10:25:20 UTC
Participant2: 10:25:06 UTC
Participant3: 10:25:24 UTC


In [16]:
elan = pd.read_csv("/content/drive/MyDrive/thesis/data/group_7/elan/Group_7.csv", header=None)
print("Tiers:", elan[0].unique().tolist())
print("\nLabels:")
for l in sorted(elan[8].dropna().str.strip().unique().tolist()):
    print(f"  {l}")

Tiers: ['Arda', 'Adam', 'Kadin', 'Arda Adam', 'Adam Kadın', 'Kadın Arda', 'Whole Group']

Labels:
  carrying_tray
  co_building_piece
  co_building_puzzle
  co_building_sub_piece
  co_building_subpiece
  co_bulding_subpiece
  co_carrying_tray
  co_inspecting_pieces
  co_merging_pieces
  co_merging_sub_pieces
  co_merging_subpiece
  co_merging_subpieces
  inscpecting_pieces
  inscpeing_target_image + traveling_between_units
  inspecitng_other_units
  inspecitng_other_untis
  inspecitng_target_image
  inspecting_other_pieces
  inspecting_other_unit
  inspecting_other_units
  inspecting_pieces
  inspecting_target_image
  inspecting_target_imge + putting_down_target_image
  matching_pieces_to_image
  matching_pieces_to_target_image
  merging_subpieces
  moving_pieces_from_table_to_tray
  moving_pieces_from_tray_to_central_table
  moving_pieces_to_tray
  non_task_convo
  object_handover
  object_handover + task_operational_convo
  picking_up_piece
  picking_up_piece + traveling_between_unit

In [24]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

BASE         = "/content/drive/MyDrive/thesis/data/group_7/"
ELAN_FILE    = BASE + "elan/Group_7_clean.csv"
PARTICIPANTS = ["Participant1", "Participant2", "Participant3"]

VIDEO_START_UTC = datetime(2026, 4, 29, 10, 24,  0, tzinfo=timezone.utc).timestamp()
XSENS_START_UTC = datetime(2026, 4, 29, 10, 24, 59, tzinfo=timezone.utc).timestamp()
XSENS_OFFSET    = XSENS_START_UTC - VIDEO_START_UTC  # +59s

ALL_TIERS = ["Participant1", "Participant2", "Participant3",
             "Participant1_Participant2", "Participant1_Participant3",
             "Participant2_Participant3", "Whole_Group"]

collab_cutoff = {
    "Participant1": 2574.0,
    "Participant2": 1041.0,
    "Participant3": 1822.0,
}

# ── ELAN cleaning ─────────────────────────────────────────────────────────────
elan_raw = pd.read_csv(BASE + "elan/Group_7.csv", header=None,
                       names=["tier","_","t_start_hms","t_start_s",
                              "t_end_hms","t_end_s","duration_hms","duration_s","label"])

tier_map = {
    "Kadın":      "Participant1",
    "Arda":       "Participant2",
    "Adam":       "Participant3",
    "Kadın Arda": "Participant1_Participant2",
    "Arda Adam":  "Participant2_Participant3",
    "Adam Kadın": "Participant1_Participant3",
    "Whole Group":"Whole_Group",
}
elan_raw["tier"] = elan_raw["tier"].str.strip().replace(tier_map)

typo_map = {
    # traveling
    "traveling_between_pieces":          "traveling_between_units",
    "traveling_beween_units":            "traveling_between_units",
    # inspecting
    "inscpecting_pieces":                "inspecting_pieces",
    "inspecitng_other_units":            "inspecting_other_units",
    "inspecitng_other_untis":            "inspecting_other_units",
    "inspecting_other_unit":             "inspecting_other_units",
    "inspecting_other_pieces":           "inspecting_other_units",
    "inspecitng_target_image":           "inspecting_target_image",
    # picking up
    "picking_up_pieces":                 "picking_up_piece",
    "picking_up_target_i":               "picking_up_target_image",
    "picking_up_traget_image":           "picking_up_target_image",
    # putting down
    "putting_doen_target_image":         "putting_down_target_image",
    "putting_down_image":                "putting_down_target_image",
    "putting_down_pieces":               "putting_down_piece",
    # co-building
    "co_building_piece":                 "co_building_subpiece",
    "co_building_puzzle":                "co_building_subpiece",
    "co_building_sub_piece":             "co_building_subpiece",
    "co_bulding_subpiece":               "co_building_subpiece",
    # co-merging
    "co_merging_pieces":                 "co_merging_subpiece",
    "co_merging_sub_pieces":             "co_merging_subpiece",
    "co_merging_subpieces":              "co_merging_subpiece",
    # merging
    "merging_subpieces":                 "merging_subpiece",
    # moving
    "moving_pieces_from_table_to_tray":  "moving_pieces_to_tray",
    "moving_pieces_from_tray_to_central_table": "moving_pieces_to_central_table_from_tray",
    # matching
    "matching_pieces_to_image":          "matching_pieces_to_target_image",
    # presenting
    "pointing_to_target_image":          "presenting_target_image",
    # searching
    "searching_for_piece":               "searching_piece",
    # convo
    "task_operaitonal_convo":            "task_operational_convo",
    "task_socail_convo":                 "task_social_convo",
}
elan_raw["label"] = elan_raw["label"].str.strip().replace(typo_map)

compound_map = {
    "inscpeing_target_image + traveling_between_units":          ["inspecting_target_image","traveling_between_units"],
    "inspecting_target_imge + putting_down_target_image":        ["inspecting_target_image","putting_down_target_image"],
    "object_handover + task_operational_convo":                  ["object_handover","task_operational_convo"],
    "picking_up_piece + traveling_between_units":                ["picking_up_piece","traveling_between_units"],
    "picking_up_target_image + traveling_between_units":         ["picking_up_target_image","traveling_between_units"],
    "putting_down_image + traveling_between_units":              ["putting_down_target_image","traveling_between_units"],
    "putting_down_target_image + traveling_between_units":       ["putting_down_target_image","traveling_between_units"],
    "task_operaitonal_convo + moving_pieces_from_table_to_tray": ["task_operational_convo","moving_pieces_to_tray"],
    "task_operational_convo + co_inspecting_pieces":             ["task_operational_convo","co_inspecting_pieces"],
    "traveling_between_units + inspecting_other_units":          ["traveling_between_units","inspecting_other_units"],
    "traveling_between_units + inspecting_target_image":         ["traveling_between_units","inspecting_target_image"],
}
expanded = []
for _, row in elan_raw.iterrows():
    label = str(row["label"]).strip()
    if label in compound_map:
        for l in compound_map[label]:
            new_row = row.copy(); new_row["label"] = l; expanded.append(new_row)
    else:
        expanded.append(row)
elan_raw = pd.DataFrame(expanded).reset_index(drop=True)

# normalize sync move
sync_map = {
    "synchronizaiton_move":              "sync_move",
    "synchronaziton_move":               "sync_move",
    "clap_synchronizaiton_move":         "sync_move",
    "khalil_synchornizaion_move":        "sync_move",
    "droping_erarble_syncornaziton_move":"sync_move",
}
elan_raw["label"] = elan_raw["label"].replace(sync_map)

# individual_building gaps with sync_move exclusion
sync_intervals = elan_raw[elan_raw["label"] == "sync_move"][["t_start_s","t_end_s"]]

new_rows = []
for participant, cutoff in collab_cutoff.items():
    p_elan = elan_raw[elan_raw["tier"] == participant].sort_values("t_start_s")
    covered = []
    for _, row in p_elan.iterrows():
        start = max(0, row["t_start_s"]); end = min(cutoff, row["t_end_s"])
        if start < end: covered.append([start, end])
    for _, row in sync_intervals.iterrows():
        start = max(0, row["t_start_s"]); end = row["t_end_s"]
        if start < end: covered.append([start, end])
    merged_iv = []
    for start, end in sorted(covered):
        if merged_iv and start <= merged_iv[-1][1]: merged_iv[-1][1] = max(merged_iv[-1][1], end)
        else: merged_iv.append([start, end])
    cursor = 0.0
    for start, end in merged_iv:
        if start > cursor:
            new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": start, "label": "individual_building"})
        cursor = max(cursor, end)
    if cursor < cutoff:
        new_rows.append({"tier": participant, "t_start_s": cursor, "t_end_s": cutoff, "label": "individual_building"})

elan = pd.concat([elan_raw, pd.DataFrame(new_rows)], ignore_index=True).sort_values(["tier","t_start_s"]).reset_index(drop=True)
elan.to_csv(ELAN_FILE, index=False)
print(f"ELAN saved: {len(elan)} rows, {elan['label'].nunique()} unique labels")
print(sorted(elan["label"].dropna().unique().tolist()))

# ── OE ────────────────────────────────────────────────────────────────────────
oe_all = []
for p in PARTICIPANTS:
    merged = load_oe_all(BASE + f"openearable/{p}/", p)
    print(f"OE {p}: {len(merged)} rows")
    oe_all.append(merged)
oe = pd.concat(oe_all, ignore_index=True).sort_values(["ts","participant"]).reset_index(drop=True)
oe["t_video"] = (oe["ts"] / 1e6) - VIDEO_START_UTC

# ── Xsens ─────────────────────────────────────────────────────────────────────
xs = load_xsens(BASE + "xsens/", PARTICIPANTS, XSENS_OFFSET)

print(f"\nOE t_video:    {oe['t_video'].min():.1f} to {oe['t_video'].max():.1f}s")
print(f"Xsens t_video: {xs['t_video'].min():.1f} to {xs['t_video'].max():.1f}s")

# ── label ─────────────────────────────────────────────────────────────────────
print("\nLabelling OE:")
oe = label_df(oe, "t_video", elan, ALL_TIERS, collab_cutoff)
print("\nLabelling Xsens:")
xs = label_df(xs, "t_video", elan, ALL_TIERS, collab_cutoff)

for tier in ALL_TIERS:
    oe[tier] = oe[tier].fillna("").replace("nan", "")
    xs[tier] = xs[tier].fillna("").replace("nan", "")

oe.to_csv(BASE + "group7_oe_labelled.csv", index=False, na_rep="")
xs.to_csv(BASE + "group7_xsens_labelled.csv", index=False, na_rep="")
print(f"\nSaved group7_oe_labelled.csv    {oe.shape}")
print(f"Saved group7_xsens_labelled.csv {xs.shape}")

ELAN saved: 367 rows, 28 unique labels
['carrying_tray', 'co_building_subpiece', 'co_carrying_tray', 'co_inspecting_pieces', 'co_merging_subpiece', 'individual_building', 'inspecting_other_units', 'inspecting_pieces', 'inspecting_target_image', 'matching_pieces_to_target_image', 'merging_subpiece', 'moving_pieces_to_central_table_from_tray', 'moving_pieces_to_tray', 'non_task_convo', 'object_handover', 'picking_up_piece', 'picking_up_target_image', 'picking_up_tray', 'presenting_piece', 'presenting_target_image', 'putting_down_piece', 'putting_down_target_image', 'putting_down_tray', 'searching_piece', 'sync_move', 'task_operational_convo', 'task_social_convo', 'traveling_between_units']
OE Participant1: 148885 rows
OE Participant2: 149891 rows
OE Participant3: 34968 rows
Xsens Participant1: 89420 rows, duration 2980.7s
Xsens Participant2: 89420 rows, duration 2980.6s
Xsens Participant3: 3292 rows, duration 109.7s

OE t_video:    66.6 to 3068.7s
Xsens t_video: 59.0 to 3039.7s

Labellin

In [25]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/thesis/data/group_1/group1_oe_labelled.csv",
                 keep_default_na=False, nrows=10)
print(df.columns.tolist())
print(df[["ts","participant","ax","ay","az"]].head(10))
print(f"\nParticipants: {df['participant'].unique()}")

['ts', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'mx', 'my', 'mz', 'baro', 'bx', 'by', 'bz', 'ppg_red', 'ppg_ir', 'ppg_green', 'ppg_ambient', 'skin_temp', 'env_temp', 'participant', 't_video', 'Arda', 'Bas', 'Rachel', 'Bas_Arda', 'Arda_Rachel', 'Bas_Rachel', 'Whole_Group']
             ts participant     ax     ay     az
0  1.776759e+15        arda -7.001 -4.918  4.540
1  1.776759e+15        arda -7.001 -4.918  4.540
2  1.776759e+15        arda -8.574 -5.010  3.901
3  1.776759e+15        arda -8.574 -5.010  3.901
4  1.776759e+15      rachel -2.766  7.240  6.932
5  1.776759e+15      rachel -2.766  7.240  6.932
6  1.776759e+15      rachel -2.766  7.240  6.932
7  1.776759e+15        arda -8.523 -5.023  2.975
8  1.776759e+15        arda -8.523 -5.023  2.975
9  1.776759e+15        arda -8.523 -5.023  2.975

Participants: ['arda' 'rachel']
